# H3 mechanism: do the T-circuit heads read the note? (Qwen2.5-VL-3B)

Owner: **@SohamRShinde**. Design and reading rule pre-registered in
`docs/DECISIONS.md` **D74** before this notebook ran.

H2 showed the note attack runs through the **7 T-only heads** (ablating them
defeats it in 70.5% of objects) and not through the 5 heads shared with P.
Ablation says those heads are *needed*; it does not say *how* they get the
note's colour. This measures, at the last token, for every head in layers 20-35:

| measure | question |
|---|---|
| **attention mass on the note's image tokens** | do they look at the note? (test **A**) |
| **direct effect** (logit lens on the head's own output) | do they write the note's colour into the answer? (test **W**) |
| attention mass on the object's tokens, all image tokens | where else do they look? |

Each value is compared with **D73's 100 layer-matched random head sets**, so
H2 and H3 judge against the same heads.

**Runs per object.** T (244 objects): incongruent and neutral notes in
3 layouts (`on_object_small` is discovery's; the other two are plan 3.7's
spatial analysis), plus the original image with no note. P (73 objects):
Prior and Visual prompts on the counterfactual image. Visual prompt
everywhere except P's Prior run. 1,854 runs, each checkpointed per object (D44).

**Eager attention.** sdpa returns no attention weights, so the model is loaded
with `attn_implementation="eager"`. Section 2 proves the captured weights are
the ones the model used (weights x values must rebuild each head's output)
and stops the notebook otherwise.

**Runtime ~2 h. Run it as Save Version -> Save & Run All (Commit).**
Section 2 times the real steps and prints the estimate.

**To resume after a cut-off:** attach the previous version's output as an
input dataset and re-run everything; section 2 copies its `h3_attention/ckpt/`
folder back and the loops skip finished objects.

**Output:** `/kaggle/working/h3_attention/`. Download only the **two** files
the last cell names (records ~5 MB gzipped). `ckpt/` holds checkpoints, not results.

Settings: GPU T4, Internet ON, Kaggle secret `HF_TOKEN` ticked under Add-ons -> Secrets.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.h3": ["FIRST_LAYER", "image_token_layout", "token_sets", "measure_run",
                           "check_attention_reconstructs_heads", "check_direct_effect_formula",
                           "capture_last_token_attention", "analyse_h3"],
    "circuit_overlap.cross_ablation": ["build_plan"],
    "circuit_overlap.notes": ["object_mask_from_pair"],
    "circuit_overlap.tokens": ["merged_grid_from_grid_thw"],
    "circuit_overlap.t_patching": ["render_t_pair", "LAYOUT", "load_patching_records",
                                   "is_t_conflicting"],
    "circuit_overlap.patching": ["LastTokenRunner"],
    "circuit_overlap.replication": ["lietzow_rows", "color_token_sets", "run_cached"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "load_notes_config",
                                 "color_first_token_ids", "ScreenConfig"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("results/replication_patching_qwen25vl3b.json", "results/t_patching_qwen25vl3b.json",
           "results/t_patching_records_qwen25vl3b.jsonl.gz"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
# Attributes the module-level check cannot see (D74 additions to shared code).
import inspect as _insp
if not _need:
    from circuit_overlap.t_screen import ScreenConfig as _SC
    from circuit_overlap.patching import LastTokenRunner as _LTR
    from circuit_overlap.t_patching import render_t_pair as _rtp
    for _what, _ok in (("ScreenConfig.attn_implementation", hasattr(_SC, "attn_implementation")),
                       ("LastTokenRunner.prefix_position_ids", "prefix_position_ids" in _insp.getsource(_LTR)),
                       ("render_t_pair note_mask", '"note_mask"' in _insp.getsource(_rtp))):
        if not _ok:
            _need[_what] = ["attribute"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, load_model
config = ScreenConfig(attn_implementation="eager")   # H3 needs attention weights (D74 (3))
model, processor = load_model(config)        # 4-bit nf4, double quant, fp16 compute on a T4 (D10)
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
assert (n_layers, n_heads) == (36, 16), "not the geometry P was discovered on"
print("attention implementation:", getattr(model.config, "_attn_implementation", "?"))

## 1b. [VERIFY] Image size and speed

Same check as `replication_patching.ipynb`: the image token budget must be
~1,280 (pinned area, as Lietzow), and a forward pass must take seconds. The
input builder is identical to P's, so the two conditions are tokenised alike.

In [ ]:
import time, torch
from datasets import load_dataset
from transformers import AutoProcessor

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_probe = ds[0]["original_image"].convert("RGB")

def build_inputs(image, prompt):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(model.device)

def n_image_tokens(image):
    t, h, w = build_inputs(image, "x")["image_grid_thw"][0].tolist()
    return t * h * w // 4                    # 2x2 patch merge -> LLM tokens

EXPECTED = config.pixels // (28 * 28)        # 1280
n_img = n_image_tokens(_probe)
print(f"image {_probe.size} -> {n_img} image tokens (expected ~{EXPECTED})")
if n_img > 1.2 * EXPECTED:
    processor = AutoProcessor.from_pretrained(
        config.checkpoint, use_fast=True,
        size={"shortest_edge": config.pixels, "longest_edge": config.pixels},
    )
    n_img = n_image_tokens(_probe)
    print(f"rebuilt the processor with size=... -> {n_img} image tokens")
assert n_img <= 1.2 * EXPECTED, "image token budget is still wrong -- do not run the loops"

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and restart"
_inp = build_inputs(_probe, VISUAL_PROMPT.format(object="banana"))
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        model(**_inp, use_cache=False)
    torch.cuda.synchronize()
    print(f"{attempt} forward pass: {time.time() - t0:.1f}s")

## 2. The plan, the objects, and checks of the measurement

The head sets and random draws come from `cross_ablation.build_plan`, exactly
as in H2 (D74 (1)). The inputs are rebuilt exactly as in discovery.

The checks stop the notebook if the measurement is wrong (D74 (10)):

1. the image tokens are one block, as many as the grid, and **row-major** by
   their own M-RoPE (row, col) positions (this settles `tokens.py`'s [VERIFY]);
2. each head's attention weights sum to 1;
3. weights x cached values **rebuild each head's `o_proj` input**: the
   weights are the ones the model used, and head h reads its own KV group;
4. the logit lens on the final residual reproduces the model's logit
   difference, and per-head outputs sum to `o_proj`'s output;
5. the cached path equals a full forward pass under eager attention.

It also saves a picture of the note and object tokens on one image
(`check_tokens.png`): look at it before trusting the masks.

In [ ]:
import glob, gzip, inflect, json, math, pathlib, shutil
from collections import Counter
import numpy as np
from PIL import Image
from circuit_overlap.replication import lietzow_rows, run_cached
from circuit_overlap.t_screen import PRIOR_PROMPT, color_first_token_ids, load_notes_config
from circuit_overlap.patching import LastTokenRunner
from circuit_overlap.t_patching import LAYOUT, render_t_pair, load_patching_records, is_t_conflicting
from circuit_overlap.cross_ablation import build_plan
from circuit_overlap.notes import object_mask_from_pair
from circuit_overlap.tokens import merged_grid_from_grid_thw
from circuit_overlap.h3 import (FIRST_LAYER, SMALL_INC, SMALL_NEU, image_token_layout, token_sets,
                                measure_run, capture_last_token_attention,
                                check_attention_reconstructs_heads, check_direct_effect_formula)

WORK = pathlib.Path("/kaggle/working/h3_attention")
CKPT = WORK / "ckpt"            # checkpoints: NOT results
CKPT.mkdir(parents=True, exist_ok=True)

# Resume: bring back checkpoints from an attached previous run's output.
found = sorted(glob.glob("/kaggle/input/**/h3_attention/ckpt", recursive=True))
if found:
    shutil.copytree(found[0], CKPT, dirs_exist_ok=True)
    print(f"resuming from {found[0]}: {sorted(p.name for p in CKPT.glob('*.jsonl'))}")

_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
row_by_i = {r["i"]: r for r in rows}
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(rows), "objects |", len(vocab), "colours | sharing a first token (D26):", collisions)

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
specs = {s.name: s for s in notes_cfg["specs"]}
neutral_words = notes_cfg["neutral_words"]
LAYOUTS = ("on_object_small", "on_object_medium", "beside_object")
assert LAYOUT == LAYOUTS[0] and all(l in specs for l in LAYOUTS), sorted(specs)

p_res = json.load(open(f"{REPO_DIR}/results/replication_patching_qwen25vl3b.json"))["result"]
t_res = json.load(open(f"{REPO_DIR}/results/t_patching_qwen25vl3b.json"))["result"]
plan = build_plan(p_res, t_res, n_heads_per_layer=n_heads)
for k, v in plan["sets"].items():
    print(f"  {k:7s} ({len(v):2d}): {', '.join(v)}")
LAYERS = list(range(FIRST_LAYER, n_layers))      # D74 (3): layers 20-35 are stored

p_disc = {r["i"]: r for r in p_res["records"] if r["status"] == "patched"}
t_disc = {r["i"]: r for r in load_patching_records(f"{REPO_DIR}/results/t_patching_records_qwen25vl3b.jsonl.gz")}
todo = {"T": [row_by_i[i] for i in t_disc], "P": [row_by_i[i] for i in p_disc]}
print(f"T {len(todo['T'])} objects, P {len(todo['P'])} (expect 244, 73)")
IMAGE_TOKEN = model.config.image_token_id

def prepared(inputs):
    """A LastTokenRunner, and the image block's offset and grid, layout-checked (D74 (10))."""
    runner = LastTokenRunner(model, inputs)
    g_rows, g_cols = merged_grid_from_grid_thw(inputs["image_grid_thw"].cpu().numpy())
    pos = runner.prefix_position_ids
    if pos is None:
        raise RuntimeError("no M-RoPE position ids were captured: cannot check the image layout")
    offset = image_token_layout(inputs["input_ids"][0].cpu().numpy(), pos[:, 0].cpu().numpy(),
                                IMAGE_TOKEN, g_rows, g_cols)
    return runner, offset, g_rows, g_cols

def answers(d):
    """(source answer, target answer) token ids from a discovery record (D74 (4))."""
    return int(d["next_token_idx_prior"]), int(d["next_token_idx_visual"])

def finish(out, d, src_run, tgt_run):
    """Status by D74 (2): same answers as discovery, everything finite."""
    a = (out["runs"][src_run]["top1"], out["runs"][tgt_run]["top1"])
    ref = [d["prior_token_logit_prior_run"], d["visual_token_logit_prior_run"],
           d["prior_token_logit_visual_run"], d["visual_token_logit_visual_run"]]
    ours = out["runs"][src_run]["pair"] + out["runs"][tgt_run]["pair"]
    out["discovery_pair_diff"] = round(max(abs(x - y) for x, y in zip(ours, ref)), 4)
    if a != answers(d):
        out.update(status="drift", discovery=list(answers(d)))
    elif not all(m["finite"] for m in out["runs"].values()):
        out["status"] = "numerics_failed"                     # plan risk 7
    else:
        out["status"] = "ok"
    return out

In [ ]:
import time, torch
VERIFY = {}
r0 = todo["T"][0]
d0 = t_disc[r0["i"]]
tok0 = answers(d0)
row0 = ds[r0["i"]]
orig0, cf0 = row0["original_image"].convert("RGB"), row0["counterfact_image"].convert("RGB")
pr0 = render_t_pair(orig0, cf0, r0["attack"], neutral_words, specs[LAYOUT])
inp0 = build_inputs(pr0["incongruent"], VISUAL_PROMPT.format(object=r0["name"]))

# [VERIFY] 1. image block: contiguous, grid-sized, row-major by M-RoPE positions.
t0_ = time.time()
runner0, offset0, gr0, gc0 = prepared(inp0)
prefill = time.time() - t0_
VERIFY["image_block"] = {"offset": offset0, "grid": [gr0, gc0], "n_tokens": int(inp0["input_ids"].shape[1])}
# [VERIFY] 5. the cached path still equals a full forward pass under eager attention.
VERIFY["cached_vs_full_eager"] = runner0.check_against_full_forward(atol=1.0)
# [VERIFY] 2. weights sum to 1 over the keys.
with capture_last_token_attention(model, LAYERS) as w0:
    runner0.logits()
VERIFY["max_weight_sum_error"] = max(float((w0[l].sum(-1) - 1).abs().max()) for l in LAYERS)
assert VERIFY["max_weight_sum_error"] < 1e-2, VERIFY
# [VERIFY] 3. weights x values rebuild every head's o_proj input slice.
VERIFY["reconstruction_rel_error"] = check_attention_reconstructs_heads(model, runner0, LAYERS, head_dim)
# [VERIFY] 4. the logit lens reproduces the model; per-head outputs sum to o_proj's.
VERIFY["direct_effect"] = check_direct_effect_formula(model, runner0, *tok0, layer=33, head_dim=head_dim)

# The measurement itself, and agreement with discovery's answers.
sets0 = token_sets(pr0["note_mask"], object_mask_from_pair(orig0, cf0), gr0, gc0, offset0)
t0_ = time.time()
m0 = measure_run(model, runner0, sets0, *tok0, LAYERS, head_dim)
measure_s = time.time() - t0_
assert m0["top1"] == tok0[0], f"{r0['name']}: eager answer {m0['top1']} vs discovery {tok0[0]}"
VERIFY["first_object"] = {"name": r0["name"], "n_tokens": m0["n_tokens"], "pair": m0["pair"],
                          "discovery_pair": [d0["prior_token_logit_prior_run"], d0["visual_token_logit_prior_run"]]}
print("first object:", VERIFY["first_object"])

# Look at this picture: note tokens red, object tokens blue, on the model's grid.
img = pr0["incongruent"].resize((gc0 * 28, gr0 * 28))
tint = np.zeros((gr0, gc0, 3), np.uint8)
for k in sets0["note"] - offset0:
    tint[k // gc0, k % gc0] = (255, 0, 0)
for k in sets0["object"] - offset0:
    tint[k // gc0, k % gc0] = (0, 0, 255)
overlay = Image.fromarray(tint).resize(img.size, Image.NEAREST)
Image.blend(img, overlay, 0.35).save(WORK / "check_tokens.png")
display(Image.open(WORK / "check_tokens.png"))

# [VERIFY] the same checks on a P object (no note; Prior prompt).
r1 = todo["P"][0]
cf1 = ds[r1["i"]]["counterfact_image"].convert("RGB")
runner1, offset1, gr1, gc1 = prepared(build_inputs(cf1, PRIOR_PROMPT.format(a_object=_eng.a(r1["name"]))))
VERIFY["reconstruction_rel_error_P"] = check_attention_reconstructs_heads(model, runner1, LAYERS, head_dim)
VERIFY["direct_effect_P"] = check_direct_effect_formula(model, runner1, *answers(p_disc[r1["i"]]),
                                                        layer=26, head_dim=head_dim)
print("checks passed:", {k: v for k, v in VERIFY.items() if k != "first_object"})

runs = 7 * len(todo["T"]) + 2 * len(todo["P"])
per_run = prefill + measure_s
VERIFY.update(prefill_s=prefill, measure_s=measure_s)
print(f"prefill {prefill:.1f} s + measure {measure_s:.2f} s per run x {runs} runs "
      f"-> ~{runs * per_run / 3600:.1f} h")
del runner0, runner1

## 3. T objects (about 2 h)

Per object: 3 layouts x (incongruent, neutral), then the original image with
no note, whose note mass is read on the `on_object_small` footprint (where the
note would be). Each layout's fool status is recorded on this path
(`is_t_conflicting`). An object whose `on_object_small` answers differ from
discovery is kept in the file but marked `drift` and not analysed.

In [ ]:
def t_object(r):
    d = t_disc[r["i"]]
    tok = answers(d)                       # (note colour, real colour)
    row = ds[r["i"]]
    original, cf = row["original_image"].convert("RGB"), row["counterfact_image"].convert("RGB")
    obj_mask = object_mask_from_pair(original, cf)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    out = {"condition": "T", "name": r["name"], "tokens": list(tok), "runs": {}, "fooled": {}}
    try:
        for layout in LAYOUTS:
            pr = render_t_pair(original, cf, r["attack"], neutral_words, specs[layout])
            if layout == LAYOUT:
                small_mask = pr["note_mask"]
            top1 = {}
            for variant in ("incongruent", "neutral"):
                runner, offset, g_rows, g_cols = prepared(build_inputs(pr[variant], prompt))
                sets = token_sets(pr["note_mask"], obj_mask, g_rows, g_cols, offset)
                out["runs"][f"{layout}/{variant}"] = measure_run(model, runner, sets, *tok, LAYERS, head_dim)
                top1[variant] = out["runs"][f"{layout}/{variant}"]["top1"]
            out["fooled"][layout] = is_t_conflicting(r, top1["neutral"], top1["incongruent"],
                                                     color_ids, ambiguous)
        runner, offset, g_rows, g_cols = prepared(build_inputs(original, prompt))
        sets = token_sets(small_mask, obj_mask, g_rows, g_cols, offset)
        out["runs"]["clean"] = measure_run(model, runner, sets, *tok, LAYERS, head_dim)
    except ValueError as exc:              # image_token_layout refused this input
        return {**out, "status": "layout_check_failed", "error": str(exc), "runs": {}}
    return finish(out, d, SMALL_INC, SMALL_NEU)

h3 = {"T": run_cached(CKPT / "h3_T.jsonl", todo["T"], t_object, progress_every=5)}
print("T", dict(Counter(r["status"] for r in h3["T"])))

## 4. P objects (about 10 min)

Prior and Visual prompts on the counterfactual image, as in discovery. These
give D74 (8)'s validation (H_P should look at the image more under the Visual
prompt, Lietzow's routing direction) and (7)'s test of whether the T-only
heads read the image's colour when there is no note.

In [ ]:
def p_object(r):
    d = p_disc[r["i"]]
    tok = answers(d)                       # (prior answer, image colour)
    row = ds[r["i"]]
    cf = row["counterfact_image"].convert("RGB")
    obj_mask = object_mask_from_pair(row["original_image"].convert("RGB"), cf)
    out = {"condition": "P", "name": r["name"], "tokens": list(tok), "runs": {}}
    prompts = {"prior": PRIOR_PROMPT.format(a_object=_eng.a(r["name"])),
               "visual": VISUAL_PROMPT.format(object=r["name"])}
    try:
        for run, prompt in prompts.items():
            runner, offset, g_rows, g_cols = prepared(build_inputs(cf, prompt))
            sets = token_sets(None, obj_mask, g_rows, g_cols, offset)
            out["runs"][run] = measure_run(model, runner, sets, *tok, LAYERS, head_dim, with_note=False)
    except ValueError as exc:
        return {**out, "status": "layout_check_failed", "error": str(exc), "runs": {}}
    return finish(out, d, "prior", "visual")

h3["P"] = run_cached(CKPT / "h3_P.jsonl", todo["P"], p_object, progress_every=10)
print("P", dict(Counter(r["status"] for r in h3["P"])))

## 5. Save

The numbers printed here are **preliminary**: the reported ones, with
intervals and D74's wording, come from `scripts/analyze_h3.py` run on the two
files saved below.

In [ ]:
from circuit_overlap.h3 import analyse_h3
from circuit_overlap.provenance import result_envelope, write_result

prelim = analyse_h3(plan, h3, n_boot=200)
for name, t in prelim["primary"].items():
    print(f"PRELIMINARY {name}: {t['observed']:+.4f} vs random {t['null_mean']:+.4f}, "
          f"p = {t['p']:.4f} -> {t['wording']}")
print("PRELIMINARY mechanism:", prelim["mechanism"])
v = prelim["validation"]["routing:P"]
print(f"PRELIMINARY validation (H_P image mass, Visual - Prior): {v['observed']:+.4f}, p = {v['p']:.4f}")

summary = {
    "plan": plan,
    "status": {c: dict(Counter(r["status"] for r in h3[c])) for c in h3},
    "verify": VERIFY,
    "layers_stored": [LAYERS[0], LAYERS[-1]],
    "records_file": "h3_attention_records_qwen25vl3b.jsonl.gz",
}
envelope = result_envelope(
    "h3_attention", summary,
    model="Qwen2.5-VL-3B-Instruct", checkpoint=config.checkpoint, seed=config.seed,
    prompt=f"T: visual={VISUAL_PROMPT!r}, incongruent vs neutral note, 3 layouts, and no note; "
           f"P: prior={PRIOR_PROMPT!r} vs visual, counterfactual image",
    n=sum(len(v) for v in h3.values()), repo_dir=REPO_DIR, decision="D74",
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0), image_tokens=n_img, layouts=list(LAYOUTS),
    attention="eager; last-token query; cached last-token path",
)
print(write_result(WORK / "h3_attention_qwen25vl3b.json", envelope))
with gzip.open(WORK / "h3_attention_records_qwen25vl3b.jsonl.gz", "wt") as f:
    for cond in ("T", "P"):
        for r in h3[cond]:
            f.write(json.dumps(r) + "\n")
print("\nDownload ONLY these two into results/ (ckpt/ and check_tokens.png are not results):")
for name in ("h3_attention_qwen25vl3b.json", "h3_attention_records_qwen25vl3b.jsonl.gz"):
    print(f"  {name:44s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")